# StormScope-EU nowcasting on MTG-I1 FCI -> Arraylake

Runs [`StormScopeMeteosatEU`](https://nvidia.github.io/earth2studio/main/modules/generated/models/px/StormScopeMeteosatEU/)
— NVIDIA's generative diffusion nowcasting model for Meteosat Third Generation FCI imagery —
end to end: fetch six consecutive FCI frames from the EUMETSAT Data Store, roll the model
forward autoregressively in 10-minute steps, **write each step to Icechunk on Arraylake as it
is produced**, and plot a true-colour RGB composite.

Based on the Earth2Studio example
[`04_stormscope_meteosat_example`](https://nvidia.github.io/earth2studio/main/examples/04_nowcasting/04_stormscope_meteosat_example/)
and the step-by-step [StormScope-EU run workflow](https://hackmd.io/@ukEyv-C_TnK1RVN9gH211w/BkSIAjzOGl).

**This needs a CUDA GPU.** The sections below run it on a Coiled GPU VM in the
`earthmover-devs` workspace.

## Run this on Coiled

Background: [Running Jupyter notebooks on a GPU with Coiled](https://docs.coiled.io/blog/jupyter-notebook-gpu.html).
Coiled installs the NVIDIA driver and CUDA runtime on the VM for you; you only supply the
Python environment.

All commands below assume credentials are loaded from `.env` first:

```bash
set -a; source .env; set +a     # EUMETSAT_CONSUMER_KEY / _SECRET, ARRAYLAKE_TOKEN
```

### 1. Build the software environment (once per region)

Already built as `earthmover-devs/stormscope-eu`. The definition lives in this repo under
`coiled/`, so rebuild it from there:

```bash
uv tool install coiled          # or: pip install "coiled[notebook]"
coiled login                    # if not already configured

coiled env create \
  --name stormscope-eu \
  --workspace earthmover-devs \
  --gpu-enabled \
  --architecture x86_64 \
  --conda coiled/environment-stormscope-eu.yml \
  --pip coiled/requirements-stormscope-eu.txt
```

**Images are built per region.** Launching in a region the env has never run in silently
queues a fresh ~9.6 GB build, and the VM gives up waiting for it:

```
Cluster status is error (reason: Scheduler Stopped -> Timed out waiting for process to phone home)
```

That is not a code error — `coiled logs <cluster-id>` for such a failure is nothing but
`Software environment has not finished building yet` on repeat. The build keeps going after
the VM dies, so wait for it and relaunch:

```bash
coiled env list --workspace earthmover-devs | grep stormscope-eu   # queued -> building -> built
```

Inspect or force a rebuild later:

```bash
coiled env inspect earthmover-devs/stormscope-eu
coiled env create ... --force-rebuild        # after editing the requirements file
```

### 2. Launch a GPU notebook session

> The subcommand is `coiled notebook start` — there is no `coiled notebook run`.
> `coiled run` is the batch/script equivalent (section 3).

```bash
coiled notebook start \
  --workspace earthmover-devs \
  --software stormscope-eu \
  --name stormscope-eu \
  --gpu \
  --vm-type g6e.xlarge --vm-type g6e.2xlarge \
  --vm-type g5.2xlarge --vm-type g5.4xlarge \
  --vm-type g6.2xlarge --vm-type g6.4xlarge \
  --region us-east-1 \
  --disk-size 200 \
  --idle-timeout '2 hours' \
  --sync \
  --allow-sync-from me \
  --env EUMETSAT_CONSUMER_KEY=$EUMETSAT_CONSUMER_KEY \
  --env EUMETSAT_CONSUMER_SECRET=$EUMETSAT_CONSUMER_SECRET \
  --env ARRAYLAKE_TOKEN=$ARRAYLAKE_TOKEN
```

JupyterLab opens in your browser. Re-running the *same* command with the same `--name`
re-opens the existing session rather than starting a second VM — that is also how you turn
`--sync` on for a session that was launched without it.

`--sync` mirrors this working directory onto the VM (needs [mutagen](https://mutagen.io/)),
so this notebook and anything it writes to `outputs/` come back to your laptop.

| flag | why |
| --- | --- |
| `--gpu` | on its own this gives Coiled's default `g4dn.xlarge` (Tesla T4, 16 GB) — too small here, so always pair it with `--vm-type`. |
| `--vm-type` (x6) | Coiled takes the first type with capacity. **Pass several.** GPU capacity is genuinely scarce: a lone `--vm-type g6e.xlarge` failed across all four `us-west-2` AZs with `insufficient capacity`. Listed largest first — `g6e` is 1x L40S (48 GB), `g5`/`g6` are A10G/L4 (24 GB) and are fine at `ensemble_size=1`. `p5.48xlarge` gets you the H100 the upstream example used. |
| `--region us-east-1` | same region as the Arraylake repo's bucket, so the Icechunk writes don't cross regions. Changing it means a per-region image rebuild — see section 1. |
| `--disk-size 200` | the env is ~9.6 GB, plus the StormScope checkpoint and cached FCI granules. |
| `--idle-timeout` | the VM shuts itself down; default is 4 hours. |
| `--env ...` | EUMETSAT and Arraylake credentials, transmitted securely (see below). |

Stop it when you're done — Coiled bills for the VM while it's up:

```bash
coiled notebook stop --name stormscope-eu --workspace earthmover-devs
```

### 3. Or run it headless as a script

`stormscope_meteosat_eu.py` is the headless twin of this notebook — same compute path, minus
the *Serve it* section.

```bash
coiled run \
  --workspace earthmover-devs \
  --software stormscope-eu \
  --name stormscope-eu-run \
  --gpu \
  --vm-type g6e.xlarge --vm-type g6e.2xlarge \
  --vm-type g5.2xlarge --vm-type g5.4xlarge \
  --vm-type g6.2xlarge --vm-type g6.4xlarge \
  --region us-east-1 \
  --disk-size 200 \
  --keepalive 1h \
  --env EUMETSAT_CONSUMER_KEY=$EUMETSAT_CONSUMER_KEY \
  --env EUMETSAT_CONSUMER_SECRET=$EUMETSAT_CONSUMER_SECRET \
  --env ARRAYLAKE_TOKEN=$ARRAYLAKE_TOKEN \
  python stormscope_meteosat_eu.py
```

**`--keepalive` matters.** By default the VM shuts down the instant the command exits, which
takes any shell you had open on it with it. Add `--detach` to return the prompt immediately
and follow the run with `coiled logs` instead of staying attached.

Killing a run mid-rollout is safe but partial: each 10-minute step is its own Icechunk commit,
so the group keeps whatever steps had committed. Re-running starts over — step 0 writes with
`mode="w"`, which wipes and recreates the group.

**Do not add `--sync` to `coiled run`.** `coiled run` uploads the script by itself, and the
mutagen session `--sync` starts is fragile: editing anything in this directory while the run
is in flight can fail it with `unable to flush session: session is not currently able to
synchronize`, which terminates the remote process mid-rollout. That is exactly how the
2026-09-08 run died after committing only the initial condition — not a model or Icechunk
error. `--sync` stays right for `coiled notebook start` in section 2, where an interactive
session is the whole point. The cost of dropping it is that nothing is copied back, so
`outputs/` exists only on the VM; read results from Arraylake instead.

**Coiled logs carry your secrets.** On a bad exit Coiled prints the full
`docker exec ... --env ...` command line, which contains the literal values of
`EUMETSAT_CONSUMER_KEY`, `EUMETSAT_CONSUMER_SECRET` and `ARRAYLAKE_TOKEN`. Do not paste that
output into Slack or an issue, and rotate the credentials if you already have.

### 4. Getting into a running VM

```bash
coiled cluster list --workspace earthmover-devs | head        # names, ids, status

coiled cluster ssh <cluster-id> --workspace earthmover-devs --dask   # software container
coiled cluster ssh <cluster-id> --workspace earthmover-devs          # host VM
coiled logs <cluster-id> --workspace earthmover-devs                 # scheduler logs
```

`--dask` puts you in the **software container**, where `python` is the `stormscope-eu` env and
the synced working directory lives at `/scratch/synced` — that is where the script runs and
`outputs/` is written. Without it you get the **host VM**, which is the level for `nvidia-smi`,
`docker` and `df`. SSH needs no setup: `coiled run` defaults to `--allow-ssh-from me`.

Because the run above set `--name stormscope-eu-run`, further `coiled run` invocations with
that name land on the *same* VM instead of starting a new one:

```bash
coiled run --name stormscope-eu-run --workspace earthmover-devs nvidia-smi
coiled run --name stormscope-eu-run --workspace earthmover-devs -it python
```

## Credentials

**EUMETSAT Data Store** (model input)

1. Register (free) at the EUMETSAT User Portal: <https://user.eumetsat.int>
2. Generate a Consumer Key / Consumer Secret pair at <https://api.eumetsat.int/api-key/>
3. Expose them as `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET`, either via
   `coiled notebook start --env ...` above, a local `.env` file (loaded below by
   `python-dotenv`), or `eumdac set-credentials <key> <secret>` which writes
   `~/.eumdac/credentials`. `MeteosatFCI` reads either.

**Arraylake** (model output)

Set `ARRAYLAKE_TOKEN` to an `ema_...` API token, or run `al auth login` once to cache
credentials in `~/.config/arraylake/`. `Client()` picks up either. On a Coiled VM there is no
browser, so pass the token via `--env`.

## What's in the environment

Two things `pip install earth2studio[stormscope,data]` will not get right on a CPU build
machine, and how `coiled/requirements-stormscope-eu.txt` handles them:

- **NATTEN** is sdist-only on PyPI, and its `setup.py` compiles the `libnatten` CUDA
  extension *only* if a GPU or `NATTEN_CUDA_ARCH` is visible at build time. Coiled builds
  software environments on CPU machines, so a PyPI install would silently produce a NATTEN
  without CUDA kernels. We pull a prebuilt wheel instead:
  `--find-links https://whl.natten.org/` + `natten==0.21.6+torch2100cu128`, matched to
  `torch==2.10.0` (whose default PyPI wheel is cu128) and Python 3.12.
- **`earth2studio[stormscope]` depends on `earth2studio[utils]`, which pulls `earth2grid`** —
  git-only, not on PyPI, and a source build. `stormscope_meteosat.py` never imports it, so we
  install the rest of the extra's contents directly (`nvidia-physicsnemo`, `nvtx`, `omegaconf`,
  `scipy`) and skip it.

`earth2studio` comes from git `main`: `StormScopeMeteosatEU` and the `MeteosatFCI` data
source are not in the 0.17.0 PyPI release. `arraylake` and `icechunk` arrive with the
`[data]` extra, so no extra pins are needed for the ingest step.

In [ ]:
import torch

import natten

print("torch     ", torch.__version__)
print("cuda avail", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device    ", torch.cuda.get_device_name(0))
print("natten    ", natten.__version__)  # 0.21.6 (installed as 0.21.6+torch2100cu128)
print("libnatten ", natten.HAS_LIBNATTEN)  # must be True, else the CUDA kernels are missing

## Set up

`StormScopeMeteosatEU` is a pure-observation nowcasting model: no NWP conditioning, the only
input is a sliding window of six consecutive FCI frames ending at the analysis time. It
operates on a fixed EU sub-region of the full MTG disk, given in native FCI pixel coordinates
by `StormScopeMeteosatEU.Model_FCI_BBox`.

In [ ]:
import os
from datetime import datetime, timezone

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()

import cartopy.crs as ccrs
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import torch
import xarray as xr
from tqdm import tqdm

import earth2studio
from earth2studio.data import fetch_data
from earth2studio.data.meteosat_fci import (
    PERSPECTIVE_POINT_HEIGHT,
    SEMI_MAJOR_AXIS,
    SEMI_MINOR_AXIS,
    MeteosatFCI,
)
from earth2studio.models.px.stormscope_meteosat import StormScopeMeteosatEU
from earth2studio.utils.coords import map_coords

from arraylake import Client
from icechunk.xarray import to_icechunk
from zarr.codecs import ZstdCodec

## Load model

`load_default_package()` downloads the checkpoint from HuggingFace on first run and caches it.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

package = StormScopeMeteosatEU.load_default_package()
model = StormScopeMeteosatEU.load_model(package=package)
model = model.to(device)
model.eval()
model.compile_model()

## Set up data sources

`MeteosatFCI` is configured with `pixel_bbox` matching the model's domain (`Model_FCI_BBox`)
so the returned `y`/`x` scan-angle coordinates align with what the model expects — no
interpolation needed.

EUMETSAT serves FCI visible and NIR channels at 1 km and IR channels at 2 km. StormScope uses
the 2 km grid for all 16 channels, so we pull from both resolutions and downsample the VIS/NIR
ones. The 1 km grid is exactly 2x the pixels of the 2 km grid, so the 1 km bounding box is the
2 km one doubled.

In [ ]:
bbox_2km = StormScopeMeteosatEU.Model_FCI_BBox
FCI_BBox = {"2km": bbox_2km}
FCI_BBox["1km"] = (
    (2 * bbox_2km[0][0], 2 * bbox_2km[0][1]),
    (2 * bbox_2km[1][0], 2 * bbox_2km[1][1]),
)
fci = {
    res: MeteosatFCI(resolution=res, pixel_bbox=bbox)
    for (res, bbox) in FCI_BBox.items()
}

## Fetch the initial condition

Six consecutive FCI frames ending at the analysis time. `fetch_data` calls the data source at
`start_time + lead_time` for each offset the model defines (`[-50 min, -40 min, ..., 0 min]`).

StormScopeMeteosatEU was trained on November 2024 – May 2026 data; pick a `start_time` outside
that window for a genuine out-of-sample test, inside it for a quick smoke test.

In [ ]:
start_time = np.datetime64(datetime(2026, 6, 30, 12, 0, 0, tzinfo=timezone.utc))
in_coords = model.input_coords()
variables = in_coords["variable"]

with torch.no_grad():
    x_res = {}
    for res in ["2km", "1km"]:
        available_vars = fci[res].available_variables()
        if res == "1km":
            # use 2km version if variable available in both 1km and 2km
            available_vars -= fci["2km"].available_variables()
        variables_res = [var for var in variables if var in available_vars]
        x_res[res] = fetch_data(
            fci[res],
            time=np.array([start_time]),
            variable=variables_res,
            lead_time=in_coords["lead_time"],
            device=device,
        )

    # 2x downsample 1km data to common 2km grid
    x, coords = StormScopeMeteosatEU.combine_1km_2km_inputs(
        *x_res["1km"], *x_res["2km"]
    )
    del x_res
    # ensure data is on model grid (reorders variables if needed)
    x, coords = map_coords(x, coords, in_coords)

print(x.shape, {k: np.shape(v) for k, v in coords.items()})

## Add the ensemble dimension

The model expects `(ensemble, time, lead_time, variable, y, x)`. Up to GPU memory limits,
`ensemble_size` can be raised to produce multiple independent members in a single forward pass.

In [ ]:
ensemble_size = 1
x = x.expand(ensemble_size, -1, -1, -1, -1, -1)
coords["ensemble"] = np.arange(ensemble_size)
coords.move_to_end("ensemble", last=False)

## Ingest target: Arraylake + Icechunk

The rollout below writes to the Icechunk repo
`earthmover-public/stormscope_eu_nvidia_gtc_berlin_2026_testing`
([web app](https://app.earthmover.io/earthmover-public/stormscope_eu_nvidia_gtc_berlin_2026_testing))
via `icechunk.xarray.to_icechunk`, one commit per 10-minute step. Writing inside the loop
rather than buffering the whole rollout is the point of the near-real-time story — and it keeps
peak host memory at one frame instead of ~3.3 GB for the full cube.

**Schema.** Dims `(ensemble, time, lead_time, y, x)`, written to the repo's **root group** —
one data variable per FCI channel (`fci06vis`, `fci105ir`, …), plus a derived `true_color`.

- `true_color` carries the extra dim `rgb` (`red`, `green`, `blue`) and holds the EUMETSAT
  true-colour recipe already applied to the three VIS channels, as 0–1 floats. Storing it
  means anything downstream — `.plot.imshow()`, a tile request, a dashboard — renders the
  composite without re-deriving it from `fci06vis`/`fci05vis`/`fci04vis`.
- `true_color_index` is the same composite folded onto a **single 0–1 scalar** with plain
  `(ensemble, time, lead_time, y, x)` dims — no `rgb` dim. Drawn with the matching
  `stormscope_true_color` colormap it looks like the RGB composite, which is what any
  consumer that can only handle one band at a time (`.plot()`, a WMS layer, an XYZ tile
  request) needs. See *One-variable true colour* below.
- `time` is the analysis (initialisation) time — one entry per model run.
- `lead_time` is the offset from it, and **only the forecast is stored**: `0 min` is the
  observed analysis frame, `+10 … +120 min` the predictions. The generator's step 0 yields the
  full initial condition — six *observed* FCI frames at `-50 … 0 min` — but the five earlier
  ones are model *input*, so `forecast_only` drops them and the axis never goes negative.
- `y`/`x` are MTG scan angles; `lat`/`lon` are attached as 2-D coordinates
  (`model.lat`, `model.lon`) so the cube is georeferenced for tiling.

**Chunking.** One chunk per 2-D field (`y`, `x` unsplit, `rgb` unsplit): the domain is
~11 MB at float32, so splitting it spatially would only multiply the object count.

**Session semantics.** `to_icechunk` takes the `Session`, not the store, and does not commit —
a session is spent once committed, so each step opens a fresh writable session.

Step 0 writes its single analysis frame with `mode="w"`, which creates the arrays and fixes
the encoding (chunking, compression) for the life of the cube — and **wipes the root group if it already has data**, so
re-running the notebook replaces the previous run rather than accumulating. Later steps use
`mode="a-"` (append only variables that carry `append_dim`), which avoids rewriting `lat`,
`lon` and the other non-`lead_time` coordinates 12 more times.

In [ ]:
ARRAYLAKE_REPO = "earthmover-public/stormscope_eu_nvidia_gtc_berlin_2026_testing"
GROUP = None  # dataset lives in the root group

# Client() reads ARRAYLAKE_TOKEN, else cached credentials from `al auth login`
client = Client()
repo = client.get_repo(ARRAYLAKE_REPO)
repo

In [ ]:
DIMS = ("ensemble", "time", "lead_time", "y", "x")
RGB_DIMS = DIMS + ("rgb",)
RGB_CHANNELS = ["fci06vis", "fci05vis", "fci04vis"]  # VIS 0.6 / 0.5 / 0.4 um

# One chunk per field: the domain is small enough (y=1120, x=2432 for the default
# Model_FCI_BBox -> ~11 MB at float32) that splitting it spatially only adds objects.
NY, NX = model.lat.shape
CHUNK = {"y": NY, "x": NX, "rgb": 3}
COMPRESSOR = ZstdCodec(level=3)

In [ ]:
def true_color(arr, variables):
    """EUMETSAT-style "true-colour" RGB composite from the three VIS channels.

    Takes the (..., variable, y, x) array of a rollout step and returns
    (..., y, x, rgb) floats in 0..1 -- display-ready, so plotting and tiling
    need no per-channel scaling.
    """
    idx = [list(variables).index(ch) for ch in RGB_CHANNELS]
    rgb = np.clip(arr[..., idx, :, :] / 24.0, 0, 1) * 255
    rgb = np.interp(
        rgb, [0, 30, 60, 120, 190, 255], [0, 110, 160, 210, 240, 255]
    ) / 255.0
    return np.moveaxis(rgb, -3, -1).astype("float32")


# --- One-variable true colour ------------------------------------------------
# The true-colour manifold is close to two-dimensional: brightness, plus a warm/cool
# split (land reflects more red than blue, water the reverse). That folds onto one
# scalar -- `true_color_index` -- with a diverging palette: 0 = darkest sea, 0.5 =
# bright cloud, 1 = darkest land. Fitted against a summer daytime scene it reproduces
# the RGB composite to ~0.03 RMSE in 0-1 colour, from a single band.
LUMA_RGB = (0.2126, 0.7152, 0.0722)  # BT.709 luminance weights
INDEX_STRETCH = (0.17, 0.90)  # luminance range of a daytime FCI EU scene

# Anchor colours: the mean composite colour per index bin of that scene.
TRUE_COLOR_PALETTE = [
    (0.000, (0.097, 0.190, 0.245)),  # deep water
    (0.082, (0.227, 0.308, 0.332)),
    (0.165, (0.356, 0.427, 0.431)),
    (0.251, (0.512, 0.546, 0.532)),
    (0.333, (0.637, 0.664, 0.647)),
    (0.416, (0.793, 0.772, 0.734)),
    (0.502, (0.955, 0.888, 0.827)),  # bright cloud top
    (0.584, (0.841, 0.767, 0.695)),
    (0.667, (0.697, 0.652, 0.598)),
    (0.749, (0.604, 0.524, 0.477)),
    (0.831, (0.493, 0.403, 0.351)),
    (0.918, (0.329, 0.284, 0.273)),
    (1.000, (0.225, 0.204, 0.198)),  # dark vegetated land
]


def true_color_index(rgb):
    """Fold the RGB composite onto a single 0-1 scalar.

    Takes (..., y, x, rgb) as returned by `true_color`, returns (..., y, x):
    the cool (water) branch runs 0 -> 0.5 with brightness, the warm (land) branch
    0.5 -> 1 the other way, so both meet at white cloud in the middle. Pair it
    with `true_color_cmap()` to get the composite back.
    """
    lo, hi = INDEX_STRETCH
    b = np.clip((rgb @ np.array(LUMA_RGB) - lo) / (hi - lo), 0, 1)
    warm = rgb[..., 0] > rgb[..., 2]  # land reflects more red than blue
    return np.where(warm, 1.0 - 0.5 * b, 0.5 * b).astype("float32")


def true_color_cmap(name="stormscope_true_color"):
    """Register and return the colormap `true_color_index` is meant to be drawn with."""
    cmap = LinearSegmentedColormap.from_list(name, TRUE_COLOR_PALETTE)
    matplotlib.colormaps.register(cmap, force=True)
    return cmap


def forecast_only(x_pred, coords_pred):
    """Trim a rollout step to `lead_time >= 0` -- the forecast, analysis frame included.

    The generator's step 0 yields the initial condition: the six *observed* FCI frames
    at -50..0 min. Only the last of them, lead 0, is part of the forecast; the five
    earlier ones are model input, so they are dropped and never reach Icechunk.
    Steps 1..N are already all-positive and pass through untouched.
    """
    lead = coords_pred["lead_time"]
    start = int(np.searchsorted(lead, np.timedelta64(0, "s")))
    if start == 0:
        return x_pred, coords_pred
    coords_out = coords_pred.copy()
    coords_out["lead_time"] = lead[start:]
    return x_pred[:, :, start:], coords_out


def prediction_to_dataset(x_pred, coords_pred, model):
    """One rollout step -> Dataset with one data variable per FCI channel, plus the
    derived `true_color` RGB composite and its single-band `true_color_index`.

    `copy=True` because the generator keeps writing into its own buffers between
    yields; without it, step 0's analysis frame would be overwritten in place.
    """
    arr = x_pred.detach().to("cpu", torch.float32, copy=True).numpy()
    data_vars = {
        str(v): (DIMS, arr[:, :, :, i]) for i, v in enumerate(coords_pred["variable"])
    }
    rgb = true_color(arr, coords_pred["variable"])
    data_vars["true_color"] = (RGB_DIMS, rgb)
    data_vars["true_color_index"] = (DIMS, true_color_index(rgb))
    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            "rgb": np.array(["red", "green", "blue"]),
            "ensemble": coords_pred["ensemble"],
            "time": coords_pred["time"],
            "lead_time": coords_pred["lead_time"],
            "y": coords_pred["y"],
            "x": coords_pred["x"],
            "lat": (("y", "x"), model.lat),
            "lon": (("y", "x"), model.lon),
        },
    )
    ds["true_color"].attrs = {
        "long_name": "true-colour RGB composite (VIS 0.6/0.5/0.4 um), 0-1",
        "source_channels": " ".join(RGB_CHANNELS),
    }
    ds["true_color_index"].attrs = {
        "long_name": "true-colour composite folded onto one band, 0-1",
        "comment": (
            "0 = darkest water, 0.5 = brightest cloud, 1 = darkest land. "
            "Render with the `stormscope_true_color` colormap over 0-1 "
            "(`raster/coolwarm` for a stock matplotlib approximation)."
        ),
        "source_channels": " ".join(RGB_CHANNELS),
    }
    ds["lat"].attrs = {"standard_name": "latitude", "units": "degrees_north"}
    ds["lon"].attrs = {"standard_name": "longitude", "units": "degrees_east"}
    ds["time"].attrs = {"long_name": "analysis (nowcast initialisation) time"}
    ds["lead_time"].attrs = {
        "long_name": (
            "offset from analysis time; 0 = observed analysis frame, "
            "> 0 = forecast. Earlier observed frames are model input and are not stored."
        )
    }
    ds.attrs = {
        "title": "StormScope-EU nowcast on MTG-I1 FCI",
        "model": "StormScopeMeteosatEU",
        "earth2studio_version": earth2studio.__version__,
        "fci_pixel_bbox": str(StormScopeMeteosatEU.Model_FCI_BBox),
    }
    return ds


def build_encoding(ds):
    """Encoding is fixed by the initial write and cannot change on append."""
    enc = {
        name: {
            "chunks": tuple(CHUNK.get(d, 1) for d in da.dims),
            "compressors": [COMPRESSOR],
            "dtype": "float32",
        }
        for name, da in ds.data_vars.items()
    }
    for name in ("lat", "lon"):
        enc[name] = {"chunks": (NY, NX), "compressors": [COMPRESSOR]}
    return enc


def lead_minutes(ds):
    return (ds["lead_time"].values / np.timedelta64(1, "m")).astype(int)

## Execute the nowcast, streaming to Icechunk

`create_iterator` handles the autoregressive rollout: the first yielded value is the
(unchanged) initial condition, each subsequent one advances the prediction by 10 minutes.
Every step is converted to a Dataset and committed before the next one is computed.

**Only the forecast is ingested.** `forecast_only` trims each step to `lead_time >= 0`, so
step 0 contributes just its analysis frame (lead 0) and the five earlier observed frames that
the model consumed as input are dropped. The cube therefore runs `0 … +N*10 min` with no
negative lead times.

In [ ]:
n_steps = 3  # 30 minutes of 10-minute forecast steps

init = np.datetime_as_string(coords["time"][0], unit="m")

for step, (x_pred, coords_pred) in enumerate(
    tqdm(model.create_iterator(x, coords), total=n_steps + 1)
):
    # Forecast only: step 0's five past observed frames are model input, not output.
    x_pred, coords_pred = forecast_only(x_pred, coords_pred)
    ds_step = prediction_to_dataset(x_pred, coords_pred, model)
    lead = lead_minutes(ds_step)

    session = repo.writable_session("main")
    if step == 0:
        # the analysis frame: the single observed frame at lead 0
        to_icechunk(
            ds_step, session, group=GROUP, mode="w", encoding=build_encoding(ds_step)
        )
        message = f"stormscope-eu {init} analysis frame (lead {lead[-1]} min)"
    else:
        to_icechunk(ds_step, session, group=GROUP, mode="a-", append_dim="lead_time")
        message = f"stormscope-eu {init} +{lead[-1]} min"
    snapshot_id = session.commit(message)
    tqdm.write(f"{message} -> {snapshot_id}")

    if step == n_steps:
        break


## Verify the round trip

Reopen the committed cube from Arraylake. `consolidated=False` is required for Icechunk
stores, and `chunks=None` skips Dask — we only want the metadata and a small slice.

In [ ]:
ds_al = xr.open_zarr(
    repo.readonly_session("main").store,
    group=GROUP,
    consolidated=False,
    chunks=None,
)

print(ds_al)
print()
print("commits:", sum(1 for _ in repo.ancestry(branch="main")))
print("web:    ", f"https://app.earthmover.io/{ARRAYLAKE_REPO}")

In [ ]:
# `true_color` is stored display-ready, so the composite is a plain slice
rgb = ds_al["true_color"].isel(ensemble=0, time=0)

In [ ]:
rgb.isel(lead_time=[0, -1]).plot.imshow(col="lead_time", rgb="rgb")

In [ ]:
# Grayscale: BT.709 luminance across the `rgb` dim. `true_color` is already
# display-encoded (the EUMETSAT stretch is baked in), so the channels are weighted
# as-is -- no linearisation. skipna=False keeps off-Earth pixels NaN.
LUMA = xr.DataArray(list(LUMA_RGB), dims="rgb", coords={"rgb": rgb["rgb"]})
gray = (rgb * LUMA).sum("rgb", skipna=False).astype("float32")
gray.attrs = {"long_name": "true-colour composite, BT.709 luminance, 0-1"}

gray.isel(lead_time=[0, -1]).plot.imshow(
    col="lead_time", cmap="Greys_r", vmin=0, vmax=1
)


### One-variable true colour

`true_color` needs the `rgb` dim, and plenty of consumers cannot carry one: `DataArray.plot()`
falls back to a colormap, a WMS `GetMap` or an XYZ tile request renders exactly one variable,
and a dashboard layer is one band. `true_color_index` is the composite folded onto a single
`(ensemble, time, lead_time, y, x)` scalar so those consumers get the same picture.

**How the fold works.** Across a scene the composite's colours lie close to a 2-D surface —
brightness, plus a warm/cool split (land reflects more red than blue, water the reverse; PCA
puts 95% of the variance on the first axis alone). Two features fold into one number by
running them in opposite directions from the middle:

```
index   0.0 ───────────── 0.5 ───────────── 1.0
        darkest water   brightest cloud   darkest land
        <── cool branch ──>  <── warm branch ──>
        s = 0.5 * brightness | s = 1 - 0.5 * brightness
```

`TRUE_COLOR_PALETTE` is then just the mean composite colour per index bin, so
`true_color_index` + `stormscope_true_color` reconstructs `true_color` to ~0.03 RMSE in 0–1
colour — visually the same image. What it costs: bright desert and bright cloud land at
similar index values, so the Sahara loses some of its orange saturation. Anything needing the
exact composite still reads `true_color`; this is the band to *serve*.

Stock matplotlib colormaps cannot do this — the palette is not a built-in — so a Flux tile or
WMS request, which only takes built-in names, gets `raster/coolwarm` as its nearest
approximation: blue water, tan land, white cloud.

In [ ]:
# Register `stormscope_true_color`, then draw the single-band variable with it.
TRUE_COLOR_CMAP = true_color_cmap()  # name: "stormscope_true_color"

tci = ds_al["true_color_index"].isel(ensemble=0, time=0)
tci.isel(lead_time=[0, -1]).plot.imshow(
    col="lead_time", cmap=TRUE_COLOR_CMAP, vmin=0, vmax=1, add_colorbar=False
)

In [ ]:
# Side by side with the three-channel composite, last frame of the rollout.
fig, axes = plt.subplots(2, 1, figsize=(11, 10))
rgb.isel(lead_time=-1).plot.imshow(ax=axes[0], rgb="rgb")
axes[0].set_title("true_color — (y, x, rgb), 3 channels")
tci.isel(lead_time=-1).plot.imshow(
    ax=axes[1], cmap=TRUE_COLOR_CMAP, vmin=0, vmax=1, add_colorbar=False
)
axes[1].set_title("true_color_index — (y, x), 1 band + stormscope_true_color")
for ax in axes:
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.tight_layout()

## Serve it: EDR, Tiles and WMS

Once the cube is committed, [Arraylake Flux](https://docs.earthmover.io/flux) serves it
directly out of Icechunk over standard geospatial protocols — no export, no second copy, and
every request is answered from a specific commit. This is the second half of the demo
narrative: the dashboard attendees open on their phones is just a slippy map pointed at the
tiles endpoint below.

Services are enabled **per organization**, not per repo, and `earthmover-public` already has
`edr`, `tiles`, `wms` and `dap2` deployments running
([service admin](https://app.earthmover.io/earthmover-public/services)). Every protocol shares
one URL grammar:

```
https://compute.earthmover.io/v1/services/{deployment}/{org}/{repo}/{ref}[/{group}]/{protocol}
```

`{deployment}` is the service's *deployment name* (defaults to the protocol name — an org can
run several per protocol, e.g. a separate `public-tiles`), `{ref}` is a branch, tag or commit
id, and `{group}` is the Zarr group holding the dataset. **We write to the root group, so
`{group}` is omitted** — that is what `GROUP = None` does in `flux_url` below.

> **Checked against a live cube** — `earthmover-public/test-nvidia-gtc`, group
> `stormscope_eu`: a different repo, but the same model output written by the same code, so
> the variable names, units and grid are identical. What that settled:
>
> - **The grid works as-is.** Flux warps the native MTG scan-angle grid to Web Mercator from
>   the 2-D `lat`/`lon` coordinates alone. No `grid_mapping` variable, no regridded EPSG:4326
>   copy, no Repo Doctor fixes — tiles come back as correctly georeferenced, limb-shaped PNGs.
> - **The channels are radiances, not brightness temperatures.** They hold FCI spectral
>   radiance in mW m⁻² sr⁻¹ (cm⁻¹)⁻¹: `fci105ir` runs roughly 20–120, the water-vapour
>   channels 0–25. A `colorscalerange` of `190,300` renders a flat, empty block — this is the
>   single easiest thing to get wrong.
> - **`lead_time`** takes `"1 hour"` or `PT1H`; a raw `120` returns 422.
> - **`true_color` cannot be composited server-side.** Flux exposes exactly two styles,
>   `raster` and `polygons`. `raster` renders one scalar band, and passing three `variables`
>   500s. The `rgb` dim is just a dimension selector, so a colour dashboard means requesting
>   `rgb=red|green|blue` as three greyscale tiles and compositing them in a canvas — or
>   showing one colormapped channel, which is what the cell below does.

In [ ]:
import io
import json

import requests

FLUX = "https://compute.earthmover.io/v1/services"
ORG, REPO = ARRAYLAKE_REPO.split("/")
REF = "main"  # any branch, tag or commit id


def flux_url(deployment, protocol=None, repo=REPO, ref=REF, group=GROUP):
    """{FLUX}/{deployment}/{org}/{repo}/{ref}[/{group}]/{protocol}

    A deployment name may carry a prefix -- `public-tiles` is a second `tiles`
    deployment that answers without auth -- so the protocol is the last segment.
    """
    protocol = protocol or deployment.split("-")[-1]
    parts = [FLUX, deployment, ORG, repo, ref]
    if group:
        parts.append(group.strip("/"))
    parts.append(protocol)
    return "/".join(parts)


# The `public-tiles` deployment answers anonymously; everything else wants a bearer
# token while the repo is private. Sending it always is harmless.
token = os.environ.get("ARRAYLAKE_TOKEN")
AUTH = {"Authorization": f"Bearer {token}"} if token else {}

for deployment in ("edr", "tiles", "public-tiles", "wms", "dap2"):
    print(f"{deployment:6s} {flux_url(deployment)}")
print()
print("service admin:", f"https://app.earthmover.io/{ORG}/services")

### EDR — point, area and cube extraction

OGC [Environmental Data Retrieval](https://ogcapi.ogc.org/edr/). Three query types hang off
the endpoint, all returning CF-CovJSON by default and CSV / GeoJSON / netCDF / Parquet /
GeoTIFF on request:

| query | selector | typical use |
| --- | --- | --- |
| `/edr/position` | `coords=POINT(lon lat)` or `MULTIPOINT(...)` | nowcast timeseries for a city |
| `/edr/area` | `coords=POLYGON((lon lat, ...))` | everything inside a catchment / country |
| `/edr/cube` | `bbox=min_x,min_y,max_x,max_y` | a subsetted regional cube |

Restrict variables with `parameter-name=<comma-separated>` and time with
`datetime=<iso>` or `datetime=<start>/<end>`; pick the encoding with `f=`.

In [ ]:
edr_url = flux_url("edr")

# Landing page: extents, available parameters, and the data_queries templates
meta = requests.get(edr_url + "/", headers=AUTH, timeout=120).json()
print(json.dumps({k: meta[k] for k in ("id", "title", "extent") if k in meta}, indent=1))
print("queries:", list(meta.get("data_queries", {})))

# --- position: a nowcast timeseries over Berlin, as CSV ---
pos = requests.get(
    edr_url + "/position",
    params={
        "coords": "POINT(13.405 52.52)",
        "parameter-name": "fci105ir,fci06vis",
        "f": "csv",  # cf_covjson (default), csv, geojson, netcdf4, parquet
    },
    headers=AUTH,
    timeout=120,
)
print(pos.status_code, pos.text[:600])

# --- cube: a lat/lon box over central Europe, as netCDF, straight into xarray ---
cube = requests.get(
    edr_url + "/cube",
    params={
        "bbox": "-5,42,20,58",  # min_x,min_y,max_x,max_y  (lon/lat)
        "parameter-name": "fci105ir",
        "f": "netcdf4",
    },
    headers=AUTH,
    timeout=300,
)
print(cube.status_code, cube.headers.get("content-type"), len(cube.content), "bytes")
if cube.ok:
    ds_edr = xr.open_dataset(io.BytesIO(cube.content))
    print(ds_edr)

# --- area: an arbitrary WKT polygon ---
# requests.get(edr_url + "/area", params={
#     "coords": "POLYGON((-5 42, 20 42, 20 58, -5 58, -5 42))",
#     "parameter-name": "fci105ir", "f": "netcdf4"}, headers=AUTH)

### Tiles — XYZ map tiles for the dashboard

This is the endpoint the GTC dashboard actually consumes: dynamically rendered PNG tiles,
reprojected to Web Mercator on the fly, no pre-baked pyramid.

```
{tiles endpoint}/{tileMatrixSet}/{z}/{y}/{x}?variables=...&style=...
```

`WebMercatorQuad` is the standard tile matrix set. **The path is `{z}/{y}/{x}`** — `y` before
`x`, which is not what MapLibre/Leaflet emit by default, so keep the template exactly as
printed rather than reordering it.

Query parameters:

| param | meaning |
| --- | --- |
| `variables` | data variable to render (repeat for vector styles) |
| `style` | `<style>/<matplotlib colormap>`. Only two styles exist, `raster` and `polygons`, and there are **no reversed (`_r`) colormaps** — for radiance, plain `raster/Greys` already puts cold cloud tops white |
| `colorscalerange` | `min,max` as a **single comma-joined string** — passing two values 422s |
| `belowmincolor` / `abovemaxcolor` | CSS colour or `transparent` for out-of-range pixels |
| `f`, `width`, `height` | `png`/`jpeg`, tile size (256) |
| *dim name* | one selector per extra dimension: `time=`, `lead_time=`, `ensemble=` |

In [ ]:
# Values are FCI spectral radiance in mW m^-2 sr^-1 (cm^-1)^-1, NOT brightness
# temperature: the 10.5 um window runs ~20-120, the water-vapour channels ~0-25.
TILE_VAR = "fci105ir"  # 10.5 um IR window — the channel the demo will show
TILE_RANGE = "20,120"  # radiance range for that channel
TILE_STYLE = "raster/Greys"  # no `_r` variants exist; Greys already puts cold = white

# `public-tiles` is the anonymous deployment. It needs no Authorization header and
# answers with `Access-Control-Allow-Origin: *`, so a browser <img>, Leaflet or MapLibre
# can load these tiles directly — which is what the dashboard needs.
TILE_DEPLOYMENT = "public-tiles"

tile_params = {
    "variables": TILE_VAR,
    "style": TILE_STYLE,
    "colorscalerange": TILE_RANGE,
    "f": "png",
    "width": 256,
    "height": 256,
    # dimension selectors — one per non-spatial dim, else the service picks index 0.
    # timedeltas take xarray-style strings ("1 hour", "30 minutes"), not ISO 8601.
    "time": np.datetime_as_string(ds_al["time"].values[0], unit="s"),
    "lead_time": "30 minutes",  # N_STEPS=3 -> the cube ends at +30 min
    "ensemble": 0,
}

# {z}/{y}/{x} — note y before x, unlike the usual {z}/{x}/{y}
tile_template = (
    flux_url(TILE_DEPLOYMENT)
    + "/WebMercatorQuad/{z}/{y}/{x}?"
    + requests.compat.urlencode(tile_params)
)
print(tile_template)

# Smoke test one tile (z=0 covers the whole world)
probe = tile_template.format(z=0, y=0, x=0)
r = requests.get(probe, headers=AUTH, timeout=120)
print(r.status_code, r.headers.get("content-type"), len(r.content), "bytes")
if r.status_code != 200:
    print(r.text[:1000])  # 422 bodies list the accepted values for a bad param

# Paste the printed URL straight into a browser address bar to see a single tile.
#
# The default `tiles` deployment would instead need `Authorization: Bearer <token>`, which
# a browser cannot attach to an <img> tile request — for that one, front the service with a
# proxy that injects the header, or use the short-lived ticketed URL the Arraylake MCP
# server's `get_tile_url` mints (valid 1 hour).

### Coastlines: the land/ocean outlines from the Earth2Studio figure

The tiles endpoint renders **only the data raster** — Flux has no basemap, no coastlines and
no labels. In the matplotlib figure at the bottom of this notebook those white outlines come
from `ax.coastlines()`, i.e. cartopy drawing Natural Earth vectors *over* the imagery.

A slippy map gets them the same way. The imagery is opaque and covers the whole viewport, so
the geography has to be a **vector layer above the tile layer**, not a basemap underneath it.
The cell below builds a standalone MapLibre page — no Python map library, because this is
exactly the shape the GTC dashboard will take — with the Flux tiles as a raster source and
Natural Earth 1:50 m `coastline` + `admin_0_boundary_lines_land` as white line layers on top.
Those are the same files cartopy ships, and `raw.githubusercontent.com` serves them with
`Access-Control-Allow-Origin: *`, so the browser can fetch them directly.

Swap `TILE_VAR` for `true_color_index` (`style=raster/coolwarm`, `colorscalerange=0,1`) for a
true-colour-ish basemap instead of a single IR channel — the tiles service renders one variable
with one built-in matplotlib colormap, which is exactly what that band is for. The overlay is
independent of what is underneath it.

In [ ]:
from IPython.display import IFrame

NE = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson"

# Leaflet and MapLibre both substitute {z}/{x}/{y} by name, so the y-before-x path
# template goes in unchanged — do not reorder it. f-string interpolation is not
# recursive, so the {z}/{y}/{x} inside tile_template survives the substitution below.
map_html = f"""<!doctype html>
<meta charset="utf-8">
<title>StormScope-EU — {TILE_VAR}</title>
<link href="https://unpkg.com/maplibre-gl@5.6.0/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@5.6.0/dist/maplibre-gl.js"></script>
<style>
  html, body {{ margin: 0; height: 100%; background: #0b0f14; }}
  #map {{ position: absolute; inset: 0; }}
</style>
<div id="map"></div>
<script>
const map = new maplibregl.Map({{
  container: "map",
  center: [8, 48],
  zoom: 3.2,
  attributionControl: {{ compact: false }},
  style: {{
    version: 8,
    sources: {{
      stormscope: {{
        type: "raster",
        tiles: ["{tile_template}"],
        tileSize: {tile_params['width']},
        attribution: "StormScope-EU (NVIDIA Earth2Studio) via Arraylake Flux"
      }},
      coastline: {{ type: "geojson", data: "{NE}/ne_50m_coastline.geojson" }},
      borders: {{ type: "geojson", data: "{NE}/ne_50m_admin_0_boundary_lines_land.geojson" }}
    }},
    layers: [
      {{ id: "bg", type: "background", paint: {{ "background-color": "#0b0f14" }} }},
      {{ id: "imagery", type: "raster", source: "stormscope" }},
      {{ id: "coastline", type: "line", source: "coastline",
        paint: {{ "line-color": "#ffffff", "line-width": 0.8 }} }},
      {{ id: "borders", type: "line", source: "borders",
        paint: {{ "line-color": "#ffffff", "line-width": 0.5, "line-opacity": 0.6 }} }}
    ]
  }}
}});
map.addControl(new maplibregl.NavigationControl());
map.on("error", e => console.log("[map-error]", e.error && e.error.message));
</script>
"""

with open("outputs/tiles_map.html", "w") as f:
    f.write(map_html)
print("wrote outputs/tiles_map.html")

IFrame("outputs/tiles_map.html", width="100%", height=600)

### WMS — map images for GIS clients

Same renderer as the tiles service, wrapped in the OGC WMS protocol, so QGIS / ArcGIS /
Leaflet's `L.tileLayer.wms` can consume it directly. `GetCapabilities` advertises one
`<Layer>` per data variable, and the same `raster/<colormap>` style list.

Add the WMS endpoint to QGIS with *Layer -> Add Layer -> Add WMS/WMTS Layer*, using the URL
printed below.

In [ ]:
wms_url = flux_url("wms")

# 1. GetCapabilities: layer names (= data variables), styles, dimensions
caps = requests.get(
    wms_url,
    params={"service": "WMS", "version": "1.3.0", "request": "GetCapabilities"},
    headers=AUTH,
    timeout=120,
)
print(caps.status_code, caps.headers.get("content-type"))
print(caps.text[:2000])

# 2. GetMap: a PNG for a bbox. Note WMS 1.3.0 EPSG:4326 bbox order is
#    miny,minx,maxy,maxx (lat first); use version=1.1.1 + srs=EPSG:4326 for lon-first.
getmap = {
    "service": "WMS",
    "version": "1.3.0",
    "request": "GetMap",
    "layers": TILE_VAR,
    "styles": TILE_STYLE,
    "crs": "EPSG:4326",
    "bbox": "35,-15,65,25",  # lat_min,lon_min,lat_max,lon_max
    "width": 1024,
    "height": 768,
    "format": "image/png",
    "colorscalerange": TILE_RANGE,
    "transparent": "true",
    # dimension selectors, same names as the tiles service
    "time": np.datetime_as_string(ds_al["time"].values[0], unit="s"),
    "lead_time": "30 minutes",  # N_STEPS=3 -> the cube ends at +30 min
    "ensemble": 0,
}
print(wms_url + "?" + requests.compat.urlencode(getmap))

img = requests.get(wms_url, params=getmap, headers=AUTH, timeout=120)
print(img.status_code, img.headers.get("content-type"), len(img.content), "bytes")
if img.headers.get("content-type", "").startswith("image"):
    with open("outputs/wms_getmap.png", "wb") as f:
        f.write(img.content)
    print("wrote outputs/wms_getmap.png")
else:
    print(img.text[:1000])

### Where this goes next

- **Rolling cube.** `time` has one entry per notebook run. To accumulate init times, write a
  full-`lead_time` template for the new analysis time with `append_dim="time"`, then fill each
  step with `to_icechunk(..., region={"time": ..., "lead_time": ...})`. `to_icechunk` has no
  `compute=False`, so the template needs `ds.to_zarr(session.store, compute=False)` or the
  private `_XarrayDatasetWriter`.
- **Tiles / dashboard.** Covered in the *Serve it* section above — the open question is
  whether Flux can render the native geostationary grid, or whether the demo needs a
  regridded EPSG:4326 copy alongside it.
- **Real time.** Replace the fixed `start_time` with "latest available FCI slot" and run the
  whole notebook on a schedule; each run is one atomic set of Icechunk commits.

## Post processing

Plot the final predicted frame from the committed cube: `true_color` is stored as an
`(y, x, rgb)` 0–1 composite, so it goes straight into `imshow` in geostationary projection
with no per-channel scaling. Off-Earth pixels are NaN.

In [ ]:
# The RGB composite is already in the cube -- take the last predicted frame.
rgb_da = ds_al["true_color"].isel(ensemble=0, time=0, lead_time=-1)

proj = ccrs.Geostationary(
    central_longitude=0.0,
    satellite_height=PERSPECTIVE_POINT_HEIGHT,
    globe=ccrs.Globe(semimajor_axis=SEMI_MAJOR_AXIS, semiminor_axis=SEMI_MINOR_AXIS),
)
extent = MeteosatFCI.projection_extent(resolution="2km", pixel_bbox=bbox_2km)

aspect = (extent[1] - extent[0]) / (extent[3] - extent[2])
fig_height = 8
fig = plt.figure(figsize=(fig_height * aspect, fig_height))
ax = fig.add_subplot(1, 1, 1, projection=proj)
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

ax.coastlines(color="white", linewidth=0.8)

im = ax.imshow(rgb_da.values, transform=proj, extent=extent, origin="lower")

lead_min = int(rgb_da["lead_time"].values / np.timedelta64(1, "m"))
valid_time = (rgb_da["time"].values + rgb_da["lead_time"].values).astype("datetime64[m]")
ax.set_title(
    f"StormScopeMeteosatEU — RGB\n" f"Valid {valid_time} UTC  (+{lead_min} min)"
)

fig.tight_layout()
fig.savefig("outputs/04_stormcast_meteosat_example.png", dpi=150)
print("Saved outputs/04_stormcast_meteosat_example.png")